In [1]:
import re
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

# The notebook lives in notebooks/, so go one level up to reach data/
df = pd.read_csv("../data/processed/complaints.csv")
t = df["text"]

print(df.shape)
print(df["product"].value_counts())

(1000, 4)
product
Checking or savings account    200
Credit card                    200
Debt collection                200
Mortgage                       200
Student loan                   200
Name: count, dtype: int64


In [2]:
def upper_ratio(s):
    letters = [c for c in s if c.isalpha()]
    return sum(c.isupper() for c in letters) / max(1, len(letters))

checks = {
    "Mostly uppercase (over 50%)":          t.apply(upper_ratio) > 0.5,
    "Space inside parentheses ( x )":       t.str.contains(r"\(\s|\s\)"),
    "Double single-quote ('') or backtick": t.str.contains(r"''|`"),
    "Space before 's":                      t.str.contains(r"\s's\b"),
    "Contains $":                           t.str.contains(r"\$"),
    "Repeated word (e.g. Texas Texas)":     t.str.contains(r"\b(\w+)\s+\1\b", case=False),
    "Empty field (e.g. Amount: Account:)":  t.str.contains(r":\s+[A-Za-z ]{3,20}:"),
    "Space before comma or period":         t.str.contains(r"\s[,.]"),
    "Contains # or *":                      t.str.contains(r"[#*]"),
}

print("Share of texts containing each pattern:\n")
for name, mask in checks.items():
    print(f"{round(mask.mean() * 100, 1):>6}%  {name}")

Share of texts containing each pattern:

   0.5%  Mostly uppercase (over 50%)
   0.0%  Space inside parentheses ( x )
   0.0%  Double single-quote ('') or backtick
   0.7%  Space before 's
  47.8%  Contains $
   4.1%  Repeated word (e.g. Texas Texas)
   3.3%  Empty field (e.g. Amount: Account:)
   0.0%  Space before comma or period
   5.0%  Contains # or *


In [3]:
print("Most common openings (first 3 words):")
print(t.str.split().str[:3].str.join(" ").str.lower().value_counts().head(10))

print("\nMost common endings (last 3 words):")
print(t.str.split().str[-3:].str.join(" ").str.lower().value_counts().head(10))

Most common openings (first 3 words):
text
i am filing        93
i am submitting    43
i am a             25
to whom it         13
i am disputing     12
i am writing       10
i received a        9
i am requesting     8
i have a            8
this is a           7
Name: count, dtype: int64

Most common endings (last 3 words):
text
are in violation.        4
my credit report.        3
debt * &                 3
they called from.        2
to this matter.          2
my credit reports.       2
& omissions insurance    2
my money back.           2
paid in full.            2
not authorize this.      2
Name: count, dtype: int64


In [4]:
print("Examples of remaining paren spacing:")
for s in t[t.str.contains(r"\(\s|\s\)")].head(8):
    hit = re.search(r".{0,40}(?:\(\s|\s\)).{0,40}", s)
    print("-", hit.group(0) if hit else s[:80])

Examples of remaining paren spacing:
